# Forecast baseline y punto de reorden

Del histórico limpio a la tabla de decisión por SKU:

1. Demanda diaria higienizada (`prepare_daily_demand`)
2. Clasificación ABC del último trimestre
3. Features rolling y backtest de la media móvil 30 días
4. Punto de reorden, stock de seguridad y pedido sugerido

El forecast de esta notebook es la media móvil 30 días para todo el catálogo. En el top clase A, `predict` la sustituye por Holt-Winters (ETS); la comparación está en la notebook 04.

Por qué se rellenan ceros y se recortan picos: notebook 03.

> Requiere el paquete instalado en modo editable: `pip install -e .` desde la raíz del repo.


In [1]:
import pandas as pd

from inventario_ecommerce import config
from inventario_ecommerce.dataset import load_transactions, save_processed
from inventario_ecommerce.features import (
    build_rolling_features,
    clean_transactions,
    compute_abc_classification,
    prepare_daily_demand,
    sales_by_product_last_quarter,
)
from inventario_ecommerce.modeling.predict import build_reorder_policy, forecast_30d_baseline
from inventario_ecommerce.modeling.train import temporal_backtest_baseline

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.2f}".format)


## 1. Datos y demanda diaria

La serie de cada SKU incluye los días sin venta (cantidad = 0) desde su primera transacción. Sin eso, la media solo vería días con ticket y sesgaría la demanda al alza.


In [2]:
raw = load_transactions()
clean = clean_transactions(raw)
daily = prepare_daily_demand(clean)
rolling = build_rolling_features(daily)

latest_features = (
    rolling.sort_values("Date")
    .groupby([config.COL_STOCK_CODE, config.COL_DESCRIPTION], as_index=False)
    .tail(1)
    .reset_index(drop=True)
)

print(f"Filas raw:     {len(raw):,}")
print(f"Filas limpias: {len(clean):,}")
print(f"Días-SKU:      {len(daily):,}")
print(f"SKUs:          {latest_features[config.COL_STOCK_CODE].nunique():,}")


Filas raw:     1,067,371
Filas limpias: 1,038,067
Días-SKU:      3,322,199
SKUs:          4,906


## 2. Clasificación ABC

- **A**: hasta el 80% de las ventas acumuladas
- **B**: del 80% al 95%
- **C**: el resto


In [3]:
last_q = sales_by_product_last_quarter(clean)
abc = compute_abc_classification(last_q)
abc["ABCClass"].value_counts()


ABCClass
C    1906
B     803
A     684
Name: count, dtype: int64

## 3. Backtest temporal

Holdout: últimos 30 días. Predicción: media diaria de los 30 días anteriores al corte. El MAE se calcula sobre el calendario completo, así que incluye los días a cero.


In [4]:
sku_metrics, global_metrics = temporal_backtest_baseline(
    daily, horizon_days=30, lookback_days=30
)
global_metrics


,CutoffDate,EvalStartDate,EvalEndDate,GlobalMAE,GlobalMAPE,HorizonDays,LookbackDays
0,2011-11-09,2011-11-10,2011-12-09,4.06,142.74,30,30


## 4. Política de reorden

Supuestos del baseline (el dataset no trae stock ni lead time real):

| Parámetro | Valor |
|-----------|------:|
| Lead time | 14 días |
| Ciclo de revisión | 7 días |
| z clase A / B / C | 1.88 / 1.65 / 1.28 |

`ROP = forecast_daily × LT + z × σ_30d × √LT`

`recommended_order_qty` es la demanda del ciclo de revisión (`forecast_daily × 7`), porque no hay stock on-hand con el que restar.


In [5]:
forecast = forecast_30d_baseline(daily, lookback_days=30, horizon_days=30)
policy = build_reorder_policy(latest_features, forecast, abc)

print(f"SKUs con recomendación: {len(policy):,}")
policy.head(20)


SKUs con recomendación: 2,963


,StockCode,Description,ABCClass,TotalSales,forecast_daily,forecast_30d,forecast_model,demand_mean_30d,demand_std_30d,demand_cv_30d,lead_time_demand,safety_stock,reorder_point,target_stock,recommended_order_qty,recommendation
0,23084,RABBIT NIGHT LIGHT,A,"56,894.39",180.23,"5,407.00",ma30,180.23,81.17,0.45,"2,523.27",570.94,"3,094.21","4,355.84","1,261.63",Monitoreo diario; evitar quiebres
1,22197,POPCORN HOLDER,A,"27,002.40",177.00,"5,310.00",ma30,177.00,83.00,0.47,"2,478.00",583.82,"3,061.82","4,300.82","1,239.00",Monitoreo diario; evitar quiebres
2,22086,PAPER CHAIN KIT 50'S CHRISTMAS,A,"50,907.49",169.13,"5,074.00",ma30,169.13,82.00,0.48,"2,367.87",576.80,"2,944.66","4,128.60","1,183.93",Monitoreo diario; evitar quiebres
3,22578,WOODEN STAR CHRISTMAS SCANDINAVIAN,A,"3,596.04",133.37,"4,001.00",ma30,133.37,84.29,0.63,"1,867.13",592.91,"2,460.04","3,393.61",933.57,Monitoreo diario; evitar quiebres
4,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,A,"4,504.03",131.43,"3,943.00",ma30,131.43,87.29,0.66,"1,840.07",614.00,"2,454.06","3,374.10",920.03,Monitoreo diario; evitar quiebres
5,22577,WOODEN HEART CHRISTMAS SCANDINAVIAN,A,"3,541.09",126.60,"3,798.00",ma30,126.60,84.76,0.67,"1,772.40",596.22,"2,368.62","3,254.82",886.20,Monitoreo diario; evitar quiebres
6,84879,ASSORTED COLOUR BIRD ORNAMENT,A,"19,298.80",118.30,"3,549.00",ma30,118.30,84.27,0.71,"1,656.20",592.80,"2,249.00","3,077.10",828.10,Monitoreo diario; evitar quiebres
7,22910,PAPER CHAIN KIT VINTAGE CHRISTMAS,A,"24,317.65",114.07,"3,422.00",ma30,114.07,74.91,0.66,"1,596.93",526.92,"2,123.86","2,922.32",798.47,Monitoreo diario; evitar quiebres
8,22952,60 CAKE CASES VINTAGE CHRISTMAS,A,"7,084.41",112.73,"3,382.00",ma30,112.73,80.50,0.71,"1,578.27",566.24,"2,144.51","2,933.64",789.13,Monitoreo diario; evitar quiebres
9,85099B,JUMBO BAG RED RETROSPOT,A,"31,101.76",106.93,"3,208.00",ma30,106.93,81.31,0.76,"1,497.07",571.94,"2,069.01","2,817.54",748.53,Monitoreo diario; evitar quiebres


## 5. Artefactos

La tabla de reorden **canónica** la escribe el CLI, que además aplica ETS en clase A:

```bash
python -m inventario_ecommerce.modeling.predict
```

Aquí se guardan solo los intermedios, para no sobrescribir esa salida con la versión de media móvil.


In [6]:
save_processed(abc, "abc_last_quarter.csv")
save_processed(latest_features, "sku_rolling_features_latest.csv")
save_processed(sku_metrics, "forecast_backtest_by_sku.csv")
save_processed(global_metrics, "forecast_backtest_global.csv")

print("Carpeta:", config.PROCESSED_DATA_DIR)


Carpeta: C:\code\proyecto-a-inventario-ecommerce\data\processed


Salida del CLI: `data/processed/inventory_reorder_recommendations.csv`.

Campos clave: `ABCClass`, `forecast_30d`, `forecast_model`, `safety_stock`, `reorder_point`, `target_stock`, `recommended_order_qty`.
